# Assay Statistics Summary (Notebook 04)

This notebook summarizes the filtered PubChem assays:
- Total assays per pathogen
- Number and % of assays with ChEMBL ID
- Global % with ChEMBL ID

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Set paths
DATA_PROCESSED = Path("../data/processed")

## 1. Load filtered assay metadata (v2)

In [25]:
# Load dataframe
df = pd.read_csv(DATA_PROCESSED / "filtered_description_with_organisms_v2_REBUILT.csv")

Some assays might have more than one pathogen detected, because we have searched for taxid and assay organism.

In [26]:
# Convert Pathogen string to list
df["Pathogen_list"] = df["Pathogen"].str.split(", ")

# Find rows where the list has more than 1 pathogen
df_multi = df[df["Pathogen_list"].apply(len) > 1].copy()

# Convert the list to a comma-separated string to allow drop_duplicates
df_multi["Pathogen_str"] = df_multi["Pathogen_list"].apply(lambda x: ", ".join(x))

# Show unique (AID, Pathogen_str) pairs
multi_pathogen_assays = df_multi[["AID", "Pathogen_str"]].reset_index(drop=True)

# Display result
multi_pathogen_assays

,AID,Pathogen_str
0,1376,"Escherichia coli, Mycobacterium tuberculosis"
1,1494,"Enterobacter, Escherichia coli"
2,1800474,"Escherichia coli, Pseudomonas aeruginosa"
3,1800475,"Escherichia coli, Pseudomonas aeruginosa"
4,2061108,"Acinetobacter baumannii, Candida albicans, Esc..."
5,463099,"Escherichia coli, Pseudomonas aeruginosa"
6,463100,"Escherichia coli, Pseudomonas aeruginosa"
7,488955,"Escherichia coli, Helicobacter pylori"
8,492957,"Escherichia coli, Helicobacter pylori"
9,623916,"Escherichia coli, Helicobacter pylori"


For the time being, we will ignore these assays until manually checked for true pathogen.

In [27]:
# Get list of AIDs with multiple pathogens
aids_to_exclude = set(multi_pathogen_assays["AID"])

# Filter df to exclude those AIDs
df = df[~df["AID"].isin(aids_to_exclude)]

## 2. Count total assays per pathogen

In [29]:
summary_assays = (
    df.groupby("Pathogen")["AID"]
    .nunique()
    .reset_index(name="Total_Assays")
    .sort_values("Total_Assays", ascending=False)
    .reset_index(drop=True)
)

summary_assays

,Pathogen,Total_Assays
0,Staphylococcus aureus,33777
1,Escherichia coli,24595
2,Pseudomonas aeruginosa,14546
3,Candida albicans,13203
4,Mycobacterium tuberculosis,12744
5,Plasmodium falciparum,10190
6,Klebsiella pneumoniae,5800
7,Streptococcus pneumoniae,4362
8,Acinetobacter baumannii,3581
9,Enterobacter,2509


## 3. Count assays with ChEMBL ID

In [31]:
df["Has_ChEMBL"] = df["ChEMBLid"].notna()

summary_chembl = (
    df[df["Has_ChEMBL"]]
    .groupby("Pathogen")["AID"]
    .nunique()
    .reset_index(name="AIDs_with_ChEMBL")
    .sort_values("AIDs_with_ChEMBL", ascending=False)
)

summary_chembl

,Pathogen,AIDs_with_ChEMBL
13,Staphylococcus aureus,33750
5,Escherichia coli,24498
11,Pseudomonas aeruginosa,14456
2,Candida albicans,13106
8,Mycobacterium tuberculosis,12598
10,Plasmodium falciparum,9997
7,Klebsiella pneumoniae,5800
14,Streptococcus pneumoniae,4356
0,Acinetobacter baumannii,3576
3,Enterobacter,2509


## 4. Merge and compute percentages

In [34]:
# Merge the two summaries
df_summary = summary_assays.merge(summary_chembl, on="Pathogen", how="left")

# Fill missing values with 0 (i.e., no ChEMBL ID found)
df_summary["AIDs_with_ChEMBL"] = df_summary["AIDs_with_ChEMBL"].fillna(0).astype(int)

# Calculate percentage of assays with ChEMBL ID
df_summary["Percent_with_ChEMBL"] = (
    100 * df_summary["AIDs_with_ChEMBL"] / df_summary["Total_Assays"]
).round(1)

df_summary

,Pathogen,Total_Assays,AIDs_with_ChEMBL,Percent_with_ChEMBL
0,Staphylococcus aureus,33777,33750,99.9
1,Escherichia coli,24595,24498,99.6
2,Pseudomonas aeruginosa,14546,14456,99.4
3,Candida albicans,13203,13106,99.3
4,Mycobacterium tuberculosis,12744,12598,98.9
5,Plasmodium falciparum,10190,9997,98.1
6,Klebsiella pneumoniae,5800,5800,100.0
7,Streptococcus pneumoniae,4362,4356,99.9
8,Acinetobacter baumannii,3581,3576,99.9
9,Enterobacter,2509,2509,100.0


## 5. Global statistics

In [38]:
total_assays = df_summary["Total_Assays"].sum()
total_with_chembl = df_summary["AIDs_with_ChEMBL"].sum()
global_percentage = 100 * total_with_chembl / total_assays

print(f"Global summary:")
print(f"- Total filtered assays: {total_assays:,}")
print(f"- With ChEMBL ID: {total_with_chembl:,} ({global_percentage:.1f}%)")

Global summary:
- Total filtered assays: 129,905
- With ChEMBL ID: 129,215 (99.5%)


## 6. Llist of ChEMBL ids

In [39]:
# Filter rows with a ChEMBL ID
chembl_pairs = df[df["ChEMBLid"].notna()][["AID", "ChEMBLid"]]
chembl_pairs.head()

,AID,ChEMBLid
1,101350,CHEMBL708592
2,101351,CHEMBL708593
3,101352,CHEMBL853638
4,101353,CHEMBL708594
5,101354,CHEMBL708595


In [40]:
# Save to CSV
output_path = DATA_PROCESSED / "aid_chembl_pairs.csv"
chembl_pairs.to_csv(output_path, index=False)